Apache Spark

## What is Apache Spark?

Apache Spark is a **unified, distributed data processing engine** designed for large-scale data analytics.   

Confusion with 'Hadoop Spark':   
- Apache Spark is the actual **distributed computing framework**, the open-source project maintained by the Apache Software Foundation. 
- It's a standalone engine for large-scale data processing that can run on **various cluster managers**. 
- It was originally developed to work with Hadoop's ecosystem to replace MapReduce.
- But Spark itself is independent and can run on other platforms like Kubernetes, Mesos, or even standalone mode without any Hadoop components.
- So there's only one Spark (Apache Spark)

![image.png](attachment:image.png)

### Core Capabilities

```text
┌────────────────────────────────────────────────────────────────────┐
│                        Apache Spark                                │
│                                                                    │
│  ┌─────────────┐ ┌─────────────┐ ┌─────────────┐ ┌─────────────┐   │
│  │  Spark SQL  │ │  Spark      │ │   MLlib     │ │  GraphX     │   │
│  │  (SQL &     │ │  Streaming  │ │  (Machine   │ │  (Graph     │   │
│  │  DataFrames)│ │  (Real-time)│ │  Learning)  │ │  Processing)│   │
│  └─────────────┘ └─────────────┘ └─────────────┘ └─────────────┘   │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │                     Spark Core (RDD)                        │   │
│  └─────────────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │              Cluster Manager (YARN / K8s / Standalone)      │   │
│  └─────────────────────────────────────────────────────────────┘   │
└────────────────────────────────────────────────────────────────────┘
```
- Large-scale batch data processing
- SQL analytics via Spark SQL
- Structured Streaming for real-time data
- Graph processing
- Machine Learning via MLlib

### Key Characteristics

| Characteristic | Description |
|---------------|-------------|
| Distributed | Processes data across multiple machines |
| Fault-tolerant | Automatically recovers from failures |
| In-memory–first execution mode | Keeps data in RAM for speed |
| Highly scalable across clusters | Handles petabytes of data |
| Multi-language | Scala, Java, Python, R |
|One of the largest open-source data projects||



## Spark Application

<img src="./pic/2_spark_app.png" width=500>

### Component Responsibilities

| Component | Responsibilities |
|-----------|------------------|
| **Driver** | Runs main program, builds DAG, splits into stages/tasks, schedules work, collects results |
| **Cluster Manager** | Allocates resources for executors (YARN, Kubernetes, Standalone) |
| **Executors** (on worker nodes) | Execute tasks in parallel, store cached data in memory, report status to driver |
| **Tasks** | Smallest unit of work, typically 1 task per partition per stage |

### Execution Flow

```text
User Code → SparkContext → DAG → Stages → Tasks → Executors
```

1. User writes transformations and actions
2. SparkContext builds **DAG (Directed Acyclic Graph)**
3. DAG is divided into **stages** (at shuffle boundaries)
4. Stages are divided into **tasks** (one per partition)
5. Tasks are sent to executors for **parallel execution**





### 运行架构解析

<img src="./pic/2_spark_application.png" width=500>  

Spark 的**主从架构（Master-Slave）**，包含以下核心组件：
1. Driver（驱动程序）

   - 运行用户编写的 `main()` 函数
   - 包含 SparkContext：整个应用的入口点，负责：
     - 连接到集群管理器（Master）
     - 将应用代码转换为 DAG（有向无环图）
     - 调度任务分配给 Executor

2. Master（集群管理器）

   - 负责整个集群的资源管理和分配
   - 接收 Driver 的资源请求
   - 协调 Worker 节点上的资源分配
   - 可以是 **Spark Standalone、YARN 或 Mesos**

3. Worker（工作节点）
    每个 Worker 节点包含：
    - Executor（执行器）
      - 一个 JVM 进程，负责实际执行计算任务
      - Cache：缓存 RDD 分区数据，加速迭代计算
      - Task：最小的执行单元，每个 Task 处理一个数据分区

**数据流向**（图中虚线箭头）
- Driver → Master：请求资源
- Master → Worker：分配 Executor
- Driver ↔ Executor：发送任务、返回结果

**执行流程**

- 提交应用：Driver 中的 SparkContext 初始化，连接 Master
- 申请资源：Master 在 Worker 上启动 Executor
- 任务调度：Driver 将任务直接发送给 Executor（绕过 Master）
- 执行计算：Executor 执行 Task，结果返回 Driver

这种架构实现了计算与存储的分离，支持水平扩展和容错处理。

### Spark 作业提交模式
提交模式就是告诉 Spark：把你的程序放在哪里运行。选择不同的提交模式，本质上是在回答一个问题：**Driver 程序运行在哪里？**
这个选择会直接影响到性能、稳定性、调试便利性。

**一、核心区别：Driver 的位置**
- Client 模式：Driver 在你的笔记本/提交机器上
- Cluster 模式：Driver 在集群节点上

**二、为什么这很重要？**
1. 网络通信开销     
    Driver 需要和所有 Executor 频繁通信（发任务、收结果）：    
    ```text
        Client 模式：
        ┌──────────┐                    ┌─────────────┐
        │ 你的电脑  │ ◄──── 大量数据 ────► │  集群机房    │
        │ (Driver) │      跨网络传输      │ (Executors) │
        └──────────┘                    └─────────────┘
        问题：网络延迟高，带宽可能成为瓶颈

        Cluster 模式：
        ┌────────────────────────────────────┐
        │              集群机房               │
        │  ┌──────┐ ◄── 内网通信 ──► ┌──────┐ │
        │  │Driver│      速度快      │Exec  │ │
        │  └──────┘                 └──────┘ │
        └────────────────────────────────────┘
    ```

2. 作业稳定性
    |场景|Client 模式|Cluster 模式|
    |----|-----------|----------|
    |关掉笔记本|❌ 作业挂掉|✅ 继续运行|
    |网络断开|❌ 作业挂掉|✅ 继续运行|
    |提交后离开❌ 不行|✅ 可以|
3. 调试便利性
    |需求|Client 模式|Cluster 模式|
    |--|----------|-------------|
    |看实时日志|✅ 直接打印到终端|❌ 需要去集群查|
    |交互式操作|✅ spark-shell|❌ 不支持|
    |断点调试✅ 本地 IDE|❌ 困难|

**三、实际场景选择**

```text   
    开发阶段：
    ├── 写代码、测试 → Local 模式
    ├── 小数据验证   → YARN-Client（方便看日志）
    └── 交互式分析   → spark-shell (Client)

    生产环境：
    ├── 定时任务     → YARN-Cluster ✅
    ├── 长时间运行   → YARN-Cluster ✅
    └── 数据管道     → YARN-Cluster ✅
```

简单说：开发用 Client，上线用 Cluster。

#### 模式分类
Spark 支持多种部署模式，但更准确的分类是按集群管理器和部署模式两个维度：    
一、按**集群管理器**分类    

| 模式| 说明| 
| ---| ---| 
| Local| 本地单机模式，用于开发测试| 
| Standalone| Spark 自带的集群管理器| 
| YARN| Hadoop 的资源管理器| 
| Mesos| Apache Mesos 资源管理器| 
| Kubernetes| K8s 容器编排平台（较新）| 

二、**部署模式（Deploy Mode）**     
针对集群模式，Driver 的运行位置分为两种：
- client
- cluster

下面是一些常见组合


##### Local 模式（开发测试）
```bash
spark-submit --master local[4] app.py
```
<img src="./pic/2_spark_local_standalone.png" width=600>  

##### Standalone 模式
```bash
spark-submit --master spark://master:7077 --deploy-mode client app.py
spark-submit --master spark://master:7077 --deploy-mode cluster app.py
```

<img src="./pic/2_spark_application.png" width=400>  



##### YARN 模式（最常用于生产）
```bash
spark-submit --master yarn --deploy-mode client app.py   # yarn-client
spark-submit --master yarn --deploy-mode cluster app.py  # yarn-cluster
```
<img src="./pic/2_spark_yarn.png" width=600>  

**YARN-Client vs YARN-Cluster 对比**

| 特性 | YARN-Client | YARN-Cluster |
|------|-------------|--------------|
| Driver 位置 | 客户端本地 | 集群的 ApplicationMaster |
| 适用场景 | 开发调试 | 生产部署 |
| 网络开销 | 大（Driver 与 Executor 跨网络通信） | 小 |
| 客户端要求 | 需保持连接 | 可断开 |

架构图示意  
```text
YARN-Client:
┌─────────┐      ┌────────────────────────┐
│ Client  │      │        YARN Cluster    │
│┌───────┐│      │  ┌────┐  ┌────┐  ┌────┐│
││Driver ││◄────►│  │Exec│  │Exec│  │Exec││
│└───────┘│      │  └────┘  └────┘  └────┘│
└─────────┘      └────────────────────────┘

YARN-Cluster:
┌─────────┐      ┌────────────────────────┐
│ Client  │─────►│        YARN Cluster    │
│ (提交后) │      │┌──────┐ ┌────┐  ┌────┐ │
│ 可断开   │      ││Driver│ │Exec│  │Exec│ │
└─────────┘      │└──────┘ └────┘  └────┘ │
                 └────────────────────────┘
```

**一、YARN-Client 模式流程**     
特点：Driver 在客户端本地运行    
```text

     客户端                              YARN 集群
   ┌────────┐                    ┌─────────────────────────┐
   │        │ ──── ① 提交申请 ───→│    ResourceManager      │
   │ Driver │                    └───────────┬─────────────┘
   │        │                                │
   │ Spark  │                         ② 分配容器
   │Context │                                ↓
   │        │                    ┌─────────────────────────┐
   │        │ ←── ③ 注册 ────────│   ApplicationMaster     │
   │        │                    │   (仅负责申请资源)        │
   │        │                    └───────────┬─────────────┘
   │        │                                │
   │        │                         ④ 申请容器
   │        │                                ↓
   │        │                    ┌──────┐ ┌──────┐ ┌──────┐
   │        │ ←─ ⑤ 直接通信 ────→ │Exec 1│ │Exec 2│ │Exec 3│
   │        │    (发任务/收结果)   └──────┘ └──────┘ └──────┘
   └────────┘
```
详细步骤

| 步骤| 动作| 说明| 
| --| ----| -----| 
| ①| 客户端提交应用| Driver 在本地启动，向 RM 申请资源| 
| ②| RM 分配容器| 在某个 NodeManager 上启动 AM| 
| ③| AM 向 Driver 注册| AM 只负责资源申请，不运行 Driver| 
| ④| AM 向 RM 申请 Executor 容器| 获取计算资源| 
| ⑤| Driver 直接与 Executor 通信| 发送任务、接收结果| 

**二、YARN-Cluster 模式流程**    
特点：Driver 在集群的 AM 中运行   
```text

     客户端                              YARN 集群
   ┌────────┐                    ┌─────────────────────────┐
   │        │ ──── ① 提交JAR ───→│    ResourceManager      │
   │ 提交后  │                    └───────────┬─────────────┘
   │ 可断开  │                                │
   │        │                         ② 分配容器
   └────────┘                                ↓
                                 ┌─────────────────────────┐
                                 │   ApplicationMaster     │
                                 │ ┌─────────────────────┐ │
                                 │ │      Driver         │ │
                                 │ │   SparkContext      │ │
                                 │ └─────────────────────┘ │
                                 └───────────┬─────────────┘
                                             │
                                      ③ 申请容器
                                             ↓
                                 ┌──────┐ ┌──────┐ ┌──────┐
                                 │Exec 1│ │Exec 2│ │Exec 3│
                                 └──────┘ └──────┘ └──────┘
                                      ↑      ↑       ↑
                                      └──────┼───────┘
                                        ④ 集群内通信
    ```
详细步骤
| 步骤| 动作| 说明| 
| ---| ----| ---| 
| ①| 客户端提交 JAR/应用| 只上传程序，不运行 Driver| 
| ②| RM 启动 AM| AM 内部包含 Driver| 
| ③| AM(Driver) 申请 Executor| 向 RM 请求计算资源| 
| ④| Driver 与 Executor 通信| 全在集群内部，网络快| 

相对于 Driver 套了层AM 的壳，AM 申请资源， Driver 实际执行

### Spark 作业解析和监控

Spark 作业执行流程:  
<img src='./pic/2_spark_execution_flow.png' width=500>


#### 1️⃣ 生成逻辑查询计划（Driver）
- 把你的代码翻译成"想做什么"
    ```python
    # 你写的代码
    df.filter(col("age") > 18).select("name", "age")
    ```

    ```text
    逻辑计划（抽象描述）：
    ├── Project [name, age]
    │   └── Filter (age > 18)
    │       └── Scan table
    ```
- 只描述意图，不关心怎么实现
- 会进行**基本优化**（如谓词下推）
- 全局视角理解整个查询
    <img src='./pic/2_spark_execution_flow_1.png' width=400>

只关心RDD的状态

#### 2️⃣ 生成物理查询计划（Driver）

- 决定"具体怎么做"
    ```text
    物理计划（具体执行方式）：
    ├── 选择 Join 策略：BroadcastHashJoin vs SortMergeJoin？
    ├── 选择扫描方式：全表扫描 vs 索引扫描？
    └── 确定并行度：分多少个 partition？
    ```

- 基于**数据统计信息**选择最优策略
- 生成可执行的 **DAG**
  <img src='./pic/2_spark_execution_flow_2.png' width=400>


#### 3️⃣ 任务调度（Driver）

- 把计划拆分成小任务，分发出去
    ```text
    DAG → Stage → Task

    Job
    ├── Stage 0 (shuffle 前)
    │   ├── Task 0 → 发给 Executor 1
    │   ├── Task 1 → 发给 Executor 2
    │   └── Task 2 → 发给 Executor 3
    └── Stage 1 (shuffle 后)
        ├── Task 0 → 发给 Executor 1
        └── Task 1 → 发给 Executor 2
    ```
- 按 **shuffle 边界**划分 Stage
- 考虑**数据本地性**分配 Task
- 知道集群资源状态
  <img src='./pic/2_spark_execution_flow_3.png' width=400>


<img src='./pic//2_spark_execution_flow_complex_DAG_staging.png' width=400>

#### 4️⃣ 任务执行（Executor）

- 真正干活
    ```text
    Executor 接收到 Task 后：
    ├── 读取数据分区
    ├── 执行计算（map/filter/reduce...）
    ├── 缓存中间结果（如果需要）
    └── 返回结果给 Driver 或写入存储
    ```
- 需要并行分布式处理

一个完整例子
```python
pythonspark.read.parquet("data.parquet") \
     .filter(col("country") == "China") \
     .groupBy("city").count() \
     .write.parquet("output")
```
```text
① 逻辑计划：读取 → 过滤 → 分组聚合 → 写入

② 物理计划：
   - 使用 Parquet 列式读取（只读需要的列）
   - 谓词下推（过滤条件推到数据源）
   - 选择 HashAggregate

③ 任务调度：
   - Stage 0: 读取 + 过滤 + 局部聚合（map端）
   - Stage 1: shuffle + 全局聚合 + 写入

④ 任务执行：
   - 各 Executor 并行处理自己的数据分区
```

一句话总结：Driver 负责"规划"，Executor 负责"干活"。

## Why Spark is Fast

### 1. RDD-Based Data Abstraction

**RDD (Resilient Distributed Dataset)**:
- **Immutable** distributed collections
- Data split into logical partitions
- Parallel processing across cluster nodes

```python
# RDD partitioning example
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Distributed across 3 partitions
# Partition 1: [1, 2, 3, 4]
# Partition 2: [5, 6, 7]
# Partition 3: [8, 9, 10]

rdd = sc.parallelize(data, 3)  # 3 partitions
```

### 2. In-Memory Computing

- Intermediate results cached in RAM
- Reduced disk I/O
- Optimized for iterative workloads

```text
Traditional Disk-Based Processing:
Step 1 → Write to Disk → Read from Disk → Step 2 → Write to Disk → Step 3
         ~100ms          ~100ms                    ~100ms

Spark In-Memory Processing:
Step 1 → Memory Cache → Step 2 → Memory Cache → Step 3
         ~0.1ms         ~0.1ms
         
Speed Improvement: 100-1000x for iterative workloads
```


### 3. Execution Efficiency

- **Pipeline execution**: Multiple operations combined within stages
- **Lazy evaluation**: Operations only execute when results are needed
- **Minimized data movement**: Intelligent task placement
- **Optimized scheduling**: Tasks run where data resides



## Creating RDDs

### Method 1: Parallelizing Existing Collections

```python
# Convert in-memory Python data to RDD
# Best for: testing, learning, small datasets

data = [1, 2, 3, 4, 5]
rdd = sc.parallelize(data)

# With specified partitions
rdd = sc.parallelize(data, numSlices=4)
```

### Method 2: Loading from External Storage

```python
# From local file system
rdd = sc.textFile("file:///path/to/data.txt")

# From HDFS
rdd = sc.textFile("hdfs://namenode:9000/data/file.txt")

# From Amazon S3
rdd = sc.textFile("s3a://bucket-name/data/file.txt")

# Multiple files (wildcard)
rdd = sc.textFile("hdfs:///data/logs/*.log")
```



## ⭐️ Transformations and Actions

### Lazy Evaluation

Key Rule: **No action → No execution**

Transformations build a plan (DAG)   
Actions trigger the plan execution -> stages -> tasks   

Example:
```python
rdd.map(...)      # Just builds plan, nothing executes
   .filter(...)   # Just builds plan, nothing executes
   .count()       # ACTION! Now everything executes
```

### Transformations (Lazy, build a plan)

**Return a new RDD (or DataFrame) without executing immediately**.

```python
# Common transformations
rdd.map(lambda x: x * 2)           # Apply function to each element
rdd.filter(lambda x: x > 5)        # Keep elements matching condition
rdd.flatMap(lambda x: x.split())   # Map + flatten results
rdd.distinct()                      # Remove duplicates
rdd.union(other_rdd)               # Combine two RDDs
```

### Actions (Trigger Execution)

**Start a job and return results to driver or storage**.

```python
# Common actions
rdd.count()                        # Count elements
rdd.collect()                      # Return all elements to driver
rdd.take(n)                        # Return first n elements
rdd.first()                        # Return first element
rdd.reduce(lambda a, b: a + b)     # Aggregate elements
```



## Narrow vs Wide Transformations

### Narrow Transformations

**Definition**: **Each output partitio**n depends on **ONE input partition**

**Characteristics**:
- No shuffle required
- No network data movement
- Can execute within same stage
- Fast and efficient

```text
Narrow Transformation (map):

Partition 1 ──map──▶ Partition 1'
Partition 2 ──map──▶ Partition 2'
Partition 3 ──map──▶ Partition 3'

Each output depends only on its corresponding input
```

**Examples**: `map`, `filter`, `select`, `withColumn`, `union`

### Wide Transformations

**Definition**: Output partitions depend on **MULTIPLE input partitions**

**Characteristics**:
- Requires shuffle across network
- Creates stage boundary
- More expensive (I/O and network)

```text
Wide Transformation (groupBy):

Partition 1 ─┐
             ├──shuffle──▶ Partition 1' (all keys "A")
Partition 2 ─┤
             ├──shuffle──▶ Partition 2' (all keys "B")
Partition 3 ─┘
             └──shuffle──▶ Partition 3' (all keys "C")

Data must move between partitions based on keys
```

**Examples**: `groupBy`, `reduceByKey`, `join`, `distinct`, `repartition`



## Spark Challenges

### System Dependencies

| Challenge | Description |
|-----------|-------------|
| No built-in storage | Relies on external storage (HDFS, S3) |
| Cluster manager required | Needs YARN, Kubernetes, or Standalone for scale |
| External coordination | Depends on ZooKeeper for some features |

### Security

| Challenge | Description |
|-----------|-------------|
| Limited native security | Basic authentication and authorization |
| Platform-dependent | Security typically from: Kerberos, IAM, Ranger |
| Network security | Requires proper firewall configuration |

### Performance & Cost

| Challenge | Description |
|-----------|-------------|
| Performance tuning is hard | Complex: partitions, shuffle, skew, memory |
| Capacity management | Difficult to right-size clusters |
| Cost optimization | Challenging in cloud environments |
| Data skew | Uneven data distribution causes bottlenecks |



## Complete Spark Example

```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count

# Initialize Spark
spark = SparkSession.builder \
    .appName("Sales Analysis") \
    .getOrCreate()

# Read data (Transformation - lazy)
df = spark.read.parquet("s3://data/sales/")

# Apply transformations (all lazy)
result = (
    df
    .filter(col("year") == 2024)              # Narrow
    .filter(col("amount") > 0)                 # Narrow
    .groupBy("region", "product_category")     # Wide (shuffle)
    .agg(
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_sale"),
        count("*").alias("num_transactions")
    )
    .orderBy(col("total_sales").desc())        # Wide (shuffle)
)

# Action - triggers execution
result.show(10)

# Another action - write results
result.write.mode("overwrite").parquet("s3://output/sales_summary/")

# Stop Spark
spark.stop()
```

---



In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("pyspark-test") \
    .getOrCreate()

df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "name"])
df.show()

RuntimeError: Only remote Spark sessions using Databricks Connect are supported. Use DatabricksSession.builder to create a remote Spark session instead.
Refer to https://docs.databricks.com/dev-tools/databricks-connect.html on how to configure Databricks Connect.